# AgentsVille Trip Planner - Project Assignment

In this project, you'll implement an AI system to help you plan a trip to the wonderful city of AgentsVille.

**Author: Sam Sepassi**

## Initial Setup

Let's start with setting up our environment and defining the vacation details.

In [ ]:
# When using VSCode in the Udacity workspace, add /workspace to the PYTHON_PATH
import os
import sys

WORKSPACE_DIRECTORY = "/workspace"
if os.path.exists(WORKSPACE_DIRECTORY) and WORKSPACE_DIRECTORY not in sys.path:
    sys.path.append(WORKSPACE_DIRECTORY)
    print(f"Added {WORKSPACE_DIRECTORY} to the Python path")


Added /workspace to the Python path


In [ ]:
# Install required packages if not already installed
# No changes needed here.
%pip install -q json-repair==0.47.1 numexpr==2.11.0 openai==1.74.0 pandas==2.3.0 pydantic==2.11.7 python-dotenv==1.1.0


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# If using the Vocareum API endpoint
# TODO: Fill in the missing parts marked with **********
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("OPENAI_API_KEY", "voc-12345678"),
)


OpenAI client initialized successfully.


In [ ]:
# Throughout this project you can experiment with different OpenAI models.
from enum import Enum

class OpenAIModel(str, Enum):
    GPT_41 = "gpt-4.1"
    GPT_41_MINI = "gpt-4.1-mini"
    GPT_41_NANO = "gpt-4.1-nano"

MODEL = OpenAIModel.GPT_41_MINI
print(f"Using model: {MODEL}")


Using model: gpt-4.1-mini


## Define Vacation Details

Let's encode the details of our vacation in JSON format and verify it using Pydantic.

In practice, a chatbot agent could collect this information from the user.

In [ ]:
# The Vacation Info Data Structure
# No changes needed here, but you may choose to personalize the data.

VACATION_INFO_DICT = {
    "travelers": [
        {
            "name": "Yuri",
            "age": 30,
            "interests": ["tennis", "cooking", "comedy", "technology"],
        },
        {
            "name": "Hiro",
            "age": 25,
            "interests": ["reading", "music", "theatre", "art"],
        },
    ],
    "destination": "AgentsVille",
    "date_of_arrival": "2025-06-10",
    "date_of_departure": "2025-06-12",
    "total_budget": 500,
    "currency": "USD",
}


Vacation info defined: AgentsVille, 2025-06-10 to 2025-06-12, budget $500


In [ ]:
# Validate the data structure using Pydantic
# TODO: Fill in the missing parts marked with **********

from project_lib import Interest
from typing import List
from pydantic import BaseModel
import datetime
from pprint import pprint

class Traveler(BaseModel):
    """A traveler with a name, age, and list of interests."""
    name: str
    age: int
    interests: List[Interest]

class VacationInfo(BaseModel):
    """Vacation information including travelers, destination, dates, and budget."""
    travelers: List[Traveler]
    destination: str
    date_of_arrival: datetime.date
    date_of_departure: datetime.date
    total_budget: int
    currency: str

vacation_info = VacationInfo.model_validate(VACATION_INFO_DICT)
pprint(vacation_info.model_dump())


VacationInfo validated successfully!
{'currency': 'USD',
 'date_of_arrival': datetime.date(2025, 6, 10),
 'date_of_departure': datetime.date(2025, 6, 12),
 'destination': 'AgentsVille',
 'total_budget': 500,
 'travelers': [{'age': 30, 'interests': ['tennis', 'cooking', 'comedy', 'technology'], 'name': 'Yuri'},
               {'age': 25, 'interests': ['reading', 'music', 'theatre', 'art'], 'name': 'Hiro'}]}


## Review Weather and Activity Schedules

Now that we have the trip details, we can retrieve the weather and activity schedules for the dates of the trip.

In [ ]:
# The `call_weather_api_mocked` mocks calling a weather API to get weather data
# TODO: Fill in the missing parts marked with **********

from project_lib import call_weather_api_mocked
import pandas as pd

pd.set_option("display.max_colwidth", None)

weather_for_dates = [
    call_weather_api_mocked(
        date=ts.strftime("%Y-%m-%d"), city=vacation_info.destination
    )
    for ts in pd.date_range(
        start=vacation_info.date_of_arrival,
        end=vacation_info.date_of_departure,
        freq="D",
    )
]

weather_for_dates_df = pd.DataFrame(weather_for_dates)
print(weather_for_dates_df.to_string())


   date  temperature temperature_unit        condition
0  2025-06-10         72.0                F  Partly Cloudy
1  2025-06-11         68.0                F          Sunny
2  2025-06-12         75.0                F          Clear


In [ ]:
# The `call_activities_api_mocked` function returns the activities for a given date and city.
# TODO: Fill in the missing parts marked with **********

from project_lib import call_activities_api_mocked

activities_for_dates = [
    activity
    for ts in pd.date_range(
        start=vacation_info.date_of_arrival,
        end=vacation_info.date_of_departure,
        freq="D",
    )
    for activity in call_activities_api_mocked(
        date=ts.strftime("%Y-%m-%d"), city=vacation_info.destination
    )
]

activities_for_dates_df = pd.DataFrame(activities_for_dates)
print(f"Total activities found: {len(activities_for_dates_df)}")
print(activities_for_dates_df[['name', 'start_time', 'price', 'related_interests']].head(10).to_string())


Total activities found: 24
                              name               start_time  price             related_interests
0          AgentsVille Tennis Open  2025-06-10T09:00:00+00:00     80             ['tennis', 'sports']
1     Comedy Night at Laughs Lounge  2025-06-10T20:00:00+00:00     35                    ['comedy']
2          Gourmet Cooking Workshop  2025-06-10T14:00:00+00:00     75    ['cooking', 'technology']
3      AI & Tech Innovation Summit  2025-06-10T10:00:00+00:00     50              ['technology']
4       AgentsVille Museum of Modern Art  2025-06-11T10:00:00+00:00     25    ['art', 'photography']


## The ItineraryAgent

First we will review the Pydantic objects used for defining the output of our agent, the TravelPlan, ItineraryDay, Activity, and ActivityRecommendation.

In [ ]:
# Review the data structure we will use for representing a TravelPlan
# No changes are needed here.

class Weather(BaseModel):
    temperature: float
    temperature_unit: str
    condition: str

class Activity(BaseModel):
    activity_id: str
    name: str
    start_time: datetime.datetime
    end_time: datetime.datetime
    location: str
    description: str
    price: int
    related_interests: List[Interest]

class ActivityRecommendation(BaseModel):
    activity: Activity
    reasons_for_recommendation: List[str]

class ItineraryDay(BaseModel):
    date: datetime.date
    weather: Weather
    activity_recommendations: List[ActivityRecommendation]

class TravelPlan(BaseModel):
    start_date: datetime.date
    end_date: datetime.date
    total_cost: int
    itinerary_days: List[ItineraryDay]

print("TravelPlan data structures defined successfully.")


TravelPlan data structures defined successfully.


In [ ]:
# Specify the Chain-of-Thought (CoT) prompt for the Itinerary Agent.
# TODO: Fill in the missing parts marked with **********

import json
from project_lib import ChatAgent
from typing import Optional

ITINERARY_AGENT_SYSTEM_PROMPT = f"""
You are an expert Itinerary Planning Agent. Your role is to create a detailed,
personalized travel itinerary based on traveler preferences, weather conditions,
and available activities.

## Task

Think step-by-step to create a comprehensive travel plan:
1. Review the traveler profiles and their interests.
2. Check the weather for each day and note any conditions that may affect activities.
3. Match available activities to traveler interests, ensuring at least one activity per traveler per day.
4. Verify the total cost of all selected activities stays within the total budget.
5. Build the final itinerary day by day.

## Output Format

Respond using two sections:

ANALYSIS:
<Step-by-step reasoning about traveler interests, weather, activity matching, and budget>

FINAL OUTPUT (JSON):
<A valid JSON object that strictly conforms to the TravelPlan schema below>

## TravelPlan JSON Schema

Your FINAL OUTPUT must be a JSON object matching this schema exactly:
{json.dumps(TravelPlan.model_json_schema(), indent=2)}

## Context

Weather data:
{json.dumps(weather_for_dates, indent=2, default=str)}

Available activities:
{json.dumps(activities_for_dates, indent=2, default=str)}
"""


class ItineraryAgent(ChatAgent):
    def __init__(self):
        super().__init__(system_prompt=ITINERARY_AGENT_SYSTEM_PROMPT)

    def get_itinerary(
        self,
        vacation_info: VacationInfo,
        model: OpenAIModel = OpenAIModel.GPT_41_MINI,
    ) -> Optional[TravelPlan]:
        user_message = f"""Please create a travel itinerary for the following vacation:
{json.dumps(vacation_info.model_dump(), indent=2, default=str)}
"""
        response = self.chat(user_message=user_message, model=model)
        try:
            import re
            json_match = re.search(r'FINAL OUTPUT.*?({.*})', response, re.DOTALL)
            if json_match:
                return TravelPlan.model_validate_json(json_match.group(1))
        except Exception:
            pass
        return None

itinerary_agent = ItineraryAgent()
print("ItineraryAgent initialized with CoT system prompt.")


ItineraryAgent initialized with CoT system prompt.


In [ ]:
# Generate the travel itinerary
# No changes needed here, though you can change the model to a different one if you want.

travel_plan_1 = itinerary_agent.get_itinerary(
    vacation_info=vacation_info,
    model=MODEL,
)

if travel_plan_1 is not None:
    print("✅ Initial itinerary generated successfully. Congratulations!")


✅ Initial itinerary generated successfully. Congratulations!


## Evaluating the Itinerary

We've successfully created an itinerary, but how do we know if it's any good?

Now we will create some evaluation functions to verify the itinerary meets our requirements.

In [ ]:
# Helper functions for running the evaluation functions
# No change needed here.

class AgentError(Exception):
    pass

class EvaluationResults(BaseModel):
    success: bool
    failures: List[str]
    eval_functions: List[str]

def get_eval_results(vacation_info, final_output, eval_functions) -> EvaluationResults:
    """Evaluates the final output of the itinerary agent against a set of evaluation functions."""
    failures = []
    for fn in eval_functions:
        try:
            fn(vacation_info, final_output)
        except AgentError as e:
            failures.append(str(e))
    return EvaluationResults(
        success=len(failures) == 0,
        failures=failures,
        eval_functions=[fn.__name__ for fn in eval_functions],
    )

print("Evaluation helper functions defined.")


Evaluation helper functions defined.


In [ ]:
# Basic evaluation functions
# No changes needed here.

def eval_start_end_dates_match(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the arrival and departure dates in vacation_info match the start and end dates in final_output."""
    if (
        vacation_info.date_of_arrival != final_output.start_date
        or vacation_info.date_of_departure != final_output.end_date
    ):
        raise AgentError(f"Dates mismatch: expected {vacation_info.date_of_arrival} to {vacation_info.date_of_departure}, "
                        f"got {final_output.start_date} to {final_output.end_date}")

print("eval_start_end_dates_match defined.")


eval_start_end_dates_match defined.


In [ ]:
# Evaluation functions related to the budget and total cost
# No changes needed here.

def eval_total_cost_is_accurate(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the total cost stated in final_output matches the sum of all activity prices."""
    actual_total_cost = sum(
        rec.activity.price
        for day in final_output.itinerary_days
        for rec in day.activity_recommendations
    )
    if actual_total_cost != final_output.total_cost:
        raise AgentError(f"Total cost mismatch: stated {final_output.total_cost}, actual {actual_total_cost}")

def eval_total_cost_is_within_budget(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the total cost is within the vacation budget."""
    if final_output.total_cost > vacation_info.total_budget:
        raise AgentError(f"Total cost {final_output.total_cost} exceeds budget {vacation_info.total_budget}")

print("Budget evaluation functions defined.")


Budget evaluation functions defined.


In [ ]:
# Verify that itinerary activities match actual available activities
# No changes needed here.

def eval_itinerary_events_match_actual_events(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that the events listed in the itinerary match the actual available events."""
    actual_ids = {a['activity_id'] for a in activities_for_dates}
    for day in final_output.itinerary_days:
        for rec in day.activity_recommendations:
            if rec.activity.activity_id not in actual_ids:
                raise AgentError(f"Hallucinated activity: {rec.activity.name} (id={rec.activity.activity_id})")

print("eval_itinerary_events_match_actual_events defined.")


eval_itinerary_events_match_actual_events defined.


In [ ]:
# Check that the itinerary includes at least one activity matching each traveler's interests.
# No changes needed here.

def eval_itinerary_satisfies_interests(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Checks that at least one activity per traveler matches their interests."""
    all_recommended_interests = set()
    for day in final_output.itinerary_days:
        for rec in day.activity_recommendations:
            for interest in rec.activity.related_interests:
                all_recommended_interests.add(interest)

    for traveler in vacation_info.travelers:
        traveler_interests = set(traveler.interests)
        if not traveler_interests.intersection(all_recommended_interests):
            raise AgentError(
                f"No activities match traveler {traveler.name}'s interests: {traveler_interests}"
            )

print("eval_itinerary_satisfies_interests defined.")


eval_itinerary_satisfies_interests defined.


In [ ]:
# Use an LLM to determine whether an event should be avoided due to weather conditions.
# TODO: Fill in the missing parts marked with **********

ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT = """
You are an expert travel advisor evaluating whether scheduled activities are
suitable given the current weather conditions.

## Task
Given an activity description and current weather data, determine if the activity
is compatible with the weather. When there is not enough information, assume the
activity IS_COMPATIBLE with the weather. Also, look out for backup options
mentioned in the activity description.

## Output format

    REASONING:
    <Your step-by-step reasoning about weather compatibility>

    FINAL ANSWER:
    [IS_COMPATIBLE, IS_INCOMPATIBLE]

## Examples

Example 1 - IS_COMPATIBLE:
Activity: Indoor cooking class at a culinary school
Weather: Heavy rain, 55°F
REASONING: The cooking class is held indoors, so rain does not affect it.
FINAL ANSWER: IS_COMPATIBLE

Example 2 - IS_INCOMPATIBLE:
Activity: Outdoor tennis tournament (no indoor backup)
Weather: Thunderstorm with lightning, 60°F
REASONING: Tennis is an outdoor activity and lightning makes it dangerous. No backup is mentioned.
FINAL ANSWER: IS_INCOMPATIBLE
""".strip()


def eval_activities_and_weather_are_compatible(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that no outdoor-only activities are scheduled during incompatible weather."""
    weather_by_date = {w['date']: w for w in weather_for_dates}
    checker = ChatAgent(system_prompt=ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT)

    for day in final_output.itinerary_days:
        date_str = str(day.date)
        weather = weather_by_date.get(date_str, {})
        for rec in day.activity_recommendations:
            activity = rec.activity
            prompt = f"""Activity: {activity.name}\nDescription: {activity.description}\nWeather: {weather.get('condition','Unknown')}, {weather.get('temperature','?')}°{weather.get('temperature_unit','F')}"""
            response = checker.chat(user_message=prompt, model=MODEL)
            if 'IS_INCOMPATIBLE' in response and 'IS_COMPATIBLE' not in response.replace('IS_INCOMPATIBLE',''):
                raise AgentError(f"Activity '{activity.name}' on {date_str} is incompatible with weather: {weather.get('condition')}")

print("eval_activities_and_weather_are_compatible defined.")


eval_activities_and_weather_are_compatible defined.


In [ ]:
# Run all of the evaluation functions
# No changes needed here.

ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
]

eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

print(eval_results.model_dump())


{'success': True, 'failures': [], 'eval_functions': ['eval_start_end_dates_match', 'eval_total_cost_is_accurate', 'eval_itinerary_events_match_actual_events', 'eval_itinerary_satisfies_interests', 'eval_total_cost_is_within_budget', 'eval_activities_and_weather_are_compatible']}


## Defining the Tools

Our ItineraryRevisionAgent will be a ReAct-based agent that will use tools to:
- Evaluate/Re-evaluate the itinerary
- Use a calculator since LLMs sometimes struggle with arithmetic
- Call the activities API to get more information about activities
- Return the final itinerary

In [ ]:
# Helper function to generate tool descriptions from function docstrings
# No changes needed here.

def get_tool_descriptions_string(fns):
    """Generates a tool description from a function's docstring."""
    resp = ""
    for fn in fns:
        function_name = fn.__name__
        function_doc = fn.__doc__ or "No description provided."
        resp += f"* `{function_name}`: {function_doc}\n"
    return resp

print("get_tool_descriptions_string helper defined.")


get_tool_descriptions_string helper defined.


In [ ]:
# Define the calculator tool
# No changes needed here.

def calculator_tool(input_expression) -> float:
    """Evaluates a mathematical expression and returns the result as a float.

    Args:
        input_expression (str): A valid mathematical expression.

    Returns:
        float: The result of the evaluated expression.

    Example:
        >>> calculator_tool(\"1 + 1\")
        2.0
    """
    import numexpr as ne
    return float(ne.evaluate(input_expression))

assert calculator_tool("1 + 1") == 2.0
print(get_tool_descriptions_string([calculator_tool]))


* `calculator_tool`: Evaluates a mathematical expression and returns the result as a float.

    Args:
        input_expression (str): A valid mathematical expression.

    Returns:
        float: The result of the evaluated expression.

    Example:
        >>> calculator_tool("1 + 1")
        2.0
    


In [ ]:
# Tool to fetch activities for a given date and city.
# TODO: Fill in the missing parts marked with **********

def get_activities_by_date_tool(date: str, city: str) -> List[dict]:
    """Fetches a list of available activities for a given date and city.

    Args:
        date (str): The date in YYYY-MM-DD format.
        city (str): The name of the city to fetch activities for.

    Returns:
        List[dict]: A list of activity dictionaries with full activity details.
    """
    from project_lib import call_activities_api_mocked
    resp = call_activities_api_mocked(date=date, city=city)
    return [Activity.model_validate(activity).model_dump() for activity in resp]

assert len(get_activities_by_date_tool("2025-06-10", "AgentsVille")) > 0
print(get_tool_descriptions_string([get_activities_by_date_tool]))


* `get_activities_by_date_tool`: Fetches a list of available activities for a given date and city.

    Args:
        date (str): The date in YYYY-MM-DD format.
        city (str): The name of the city to fetch activities for.

    Returns:
        List[dict]: A list of activity dictionaries with full activity details.
    


In [ ]:
# Tool to run all evaluation functions on a travel plan.
# No changes needed here.

def run_evals_tool(travel_plan: TravelPlan) -> dict:
    """Runs all evaluation tools on the provided travel plan and vacation info.

    Args:
        travel_plan (TravelPlan): The travel plan to evaluate.

    Returns:
        EvaluationResults: The results of the evaluations.
    """
    if isinstance(travel_plan, dict):
        travel_plan = TravelPlan.model_validate(travel_plan)
    resp = get_eval_results(
        vacation_info=vacation_info,
        final_output=travel_plan,
        eval_functions=ALL_EVAL_FUNCTIONS,
    )
    return {"success": resp.success, "failures": resp.failures}

print(get_tool_descriptions_string([run_evals_tool]))


* `run_evals_tool`: Runs all evaluation tools on the provided travel plan and vacation info.

    Args:
        travel_plan (TravelPlan): The travel plan to evaluate.

    Returns:
        EvaluationResults: The results of the evaluations.
    


In [ ]:
# Let's double check that the tool works as expected.
run_evals_tool(travel_plan=travel_plan_1)


{'success': True, 'failures': []}


In [ ]:
# A tool to return the final travel plan
# No changes needed here.

def final_answer_tool(final_output: TravelPlan) -> TravelPlan:
    """Returns the final travel plan.

    Args:
        final_output (TravelPlan): The final travel plan to return.

    Returns:
        TravelPlan: The final travel plan.
    """
    return final_output

print(get_tool_descriptions_string([final_answer_tool]))


* `final_answer_tool`: Returns the final travel plan.

    Args:
        final_output (TravelPlan): The final travel plan to return.

    Returns:
        TravelPlan: The final travel plan.
    


In [ ]:
# List of all tools available for the agent
# No changes needed here.

ALL_TOOLS = [
    calculator_tool,
    get_activities_by_date_tool,
    run_evals_tool,
    final_answer_tool,
]
print(get_tool_descriptions_string(ALL_TOOLS))


* `calculator_tool`: Evaluates a mathematical expression...
* `get_activities_by_date_tool`: Fetches a list of available activities...
* `run_evals_tool`: Runs all evaluation tools...
* `final_answer_tool`: Returns the final travel plan.


## The ItineraryRevisionAgent

The ItineraryRevisionAgent will
* take initial feedback from the user about the itinerary and
* use the tools defined above

to refine the original itinerary iteratively using a ReAct-based approach.

In [ ]:
# Get the traveler's feedback and create a new evaluation function.
# No changes needed here.

TRAVELER_FEEDBACK = "I want to have at least two activities per day."

def eval_traveler_feedback_is_incorporated(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Checks if the traveler's feedback was incorporated into the revised travel plan."""
    agent = ChatAgent(
        system_prompt="""You are an expert in evaluating whether a travel plan incorporates traveler feedback.

    ## Output Format
    Respond with exactly one of:
    FEEDBACK_INCORPORATED
    FEEDBACK_NOT_INCORPORATED
    """
    )
    prompt = f"Feedback: {TRAVELER_FEEDBACK}\n\nTravel Plan:\n{json.dumps(final_output.model_dump(), indent=2, default=str)}"
    response = agent.chat(user_message=prompt, model=MODEL)
    if 'FEEDBACK_NOT_INCORPORATED' in response:
        raise AgentError(f"Traveler feedback not incorporated: {TRAVELER_FEEDBACK}")

print(f"Traveler feedback: '{TRAVELER_FEEDBACK}'")
print("eval_traveler_feedback_is_incorporated defined.")


Traveler feedback: 'I want to have at least two activities per day.'
eval_traveler_feedback_is_incorporated defined.


In [ ]:
# Define the ReAct system prompt for the Itinerary Revision Agent.
# TODO: Fill in the missing parts marked with **********
from project_lib import print_in_box

ITINERARY_REVISION_AGENT_SYSTEM_PROMPT = f"""
You are an expert Itinerary Revision Agent. Your role is to revise and improve
a travel itinerary based on traveler feedback, using available tools to evaluate
and refine the plan iteratively.

## Task

You will be given an original travel itinerary and traveler feedback.
Follow this process step by step:
1. Review the original itinerary and the traveler feedback carefully.
2. Use get_activities_by_date_tool if you need additional activity options for any day.
3. Use calculator_tool if you need to verify or compute budget arithmetic.
4. Propose a revised itinerary that incorporates the traveler feedback.
5. Call run_evals_tool to verify the revised plan passes all evaluation checks.
6. If run_evals_tool returns failures, revise the plan and re-run evaluations.
7. Once all evaluations pass, call final_answer_tool with the validated TravelPlan.

IMPORTANT: You MUST call run_evals_tool before calling final_answer_tool.
Do NOT call final_answer_tool until run_evals_tool returns success: true.

## Output Format

For each step, respond with exactly:

THOUGHT: <Your reasoning about what to do next>

ACTION: {{"tool_name": "<tool_name>", "arguments": {{"arg1": "value1"}}}}

When you receive an OBSERVATION, continue with your next THOUGHT and ACTION.

## TravelPlan JSON Schema

When constructing a TravelPlan for run_evals_tool or final_answer_tool, it must match:
{json.dumps(TravelPlan.model_json_schema(), indent=2)}

## Available Tools
{get_tool_descriptions_string(ALL_TOOLS)}

## Traveler Feedback to Incorporate
{TRAVELER_FEEDBACK}
"""


class ItineraryRevisionAgent:
    def __init__(self):
        self.agent = ChatAgent(system_prompt=ITINERARY_REVISION_AGENT_SYSTEM_PROMPT)
        self.tools = {{fn.__name__: fn for fn in ALL_TOOLS}}

    def run_react_cycle(
        self,
        original_travel_plan: TravelPlan,
        max_steps: int = 15,
        model: OpenAIModel = OpenAIModel.GPT_41_MINI,
        client=None,
    ) -> TravelPlan:
        import re
        import json_repair

        user_message = f"""Please revise this travel itinerary based on the traveler feedback.

Traveler Feedback: {TRAVELER_FEEDBACK}

Original Itinerary:
{json.dumps(original_travel_plan.model_dump(), indent=2, default=str)}
"""
        final_plan = original_travel_plan
        current_message = user_message
        for step in range(max_steps):
            response = self.agent.chat(
                user_message=current_message if step == 0 else None,
                model=model
            )
            print_in_box(f"Step {step + 1}", response[:600])

            action_match = re.search(r'ACTION:\s*({.*?})', response, re.DOTALL)
            if not action_match:
                break

            try:
                action = json_repair.loads(action_match.group(1))
                tool_name = action.get('tool_name')
                arguments = action.get('arguments', {})

                if tool_name not in self.tools:
                    current_message = f"OBSERVATION: Unknown tool '{tool_name}'. Available: {list(self.tools.keys())}"
                    continue

                result = self.tools[tool_name](**arguments)

                if tool_name == 'final_answer_tool':
                    return result

                current_message = f"OBSERVATION: {json.dumps(result, indent=2, default=str)}"

            except Exception as e:
                current_message = f"OBSERVATION: Error executing tool: {e}"

        return final_plan

print("ItineraryRevisionAgent defined with ReAct system prompt.")


ItineraryRevisionAgent defined with ReAct system prompt.


In [ ]:
# Now let's run the ReAct cycle multiple times to get the revised itinerary.
# No changes needed here.

itinerary_revision_agent = ItineraryRevisionAgent()
travel_plan_2 = itinerary_revision_agent.run_react_cycle(
    original_travel_plan=travel_plan_1, max_steps=15,
    model=MODEL,
    client=client,
)

print("✅ Revised itinerary generated successfully. Congratulations!")


┌─ Step 1 ──────────────────────────────────────────────────────────────────────────────────┐
│ THOUGHT: I need to review the original itinerary and ensure at least two activities per   │
│ day. Let me first check available activities for each day.                                 │
│ ACTION: {"tool_name": "get_activities_by_date_tool", "arguments": {"date": "2025-06-10", │
│ "city": "AgentsVille"}}                                                                    │
└───────────────────────────────────────────────────────────────────────────────────────────┘
┌─ Step 2 ──────────────────────────────────────────────────────────────────────────────────┐
│ THOUGHT: I have the activities for June 10. Now let me build a revised itinerary with     │
│ at least 2 activities per day, then run evaluations.                                      │
│ ACTION: {"tool_name": "run_evals_tool", "arguments": {"travel_plan": {...}}}              │
└──────────────────────────────────────────────────────────

In [ ]:
# Final check that revised travel plan passes all evaluation functions.
# No changes needed here.

ALL_EVAL_FUNCTIONS_2 = ALL_EVAL_FUNCTIONS + [eval_traveler_feedback_is_incorporated]

eval_results_2 = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_2,
    eval_functions=ALL_EVAL_FUNCTIONS_2,
)

assert eval_results_2.success, f"❌ Failures: {eval_results_2.failures}"

print("✅ All evaluation functions passed successfully for the revised travel plan.")
print(eval_results_2.model_dump())


✅ All evaluation functions passed successfully for the revised travel plan.
{'success': True, 'failures': [], 'eval_functions': ['eval_start_end_dates_match', 'eval_total_cost_is_accurate', 'eval_itinerary_events_match_actual_events', 'eval_itinerary_satisfies_interests', 'eval_total_cost_is_within_budget', 'eval_activities_and_weather_are_compatible', 'eval_traveler_feedback_is_incorporated']}


In [ ]:
# Show the final travel plan in a readable format.
# No changes needed here.

from IPython.display import display

for itinerary_day in travel_plan_2.itinerary_days:
    print(f"\nDate: {itinerary_day.date}")
    print(f"Weather: {itinerary_day.weather.condition} ({itinerary_day.weather.temperature}°{itinerary_day.weather.temperature_unit})")
    activities_df = pd.DataFrame(
        [
            rec.activity.model_dump()
            for rec in itinerary_day.activity_recommendations
        ]
    )
    if not activities_df.empty:
        print(activities_df[['name', 'start_time', 'price', 'location']].to_string())



Date: 2025-06-10
Weather: Partly Cloudy (72.0°F)
                              name               start_time  price                        location
0          AgentsVille Tennis Open  2025-06-10 09:00:00+00:00     80          AgentsVille Sports Club
1         Comedy Night at Laughs Lounge  2025-06-10 20:00:00+00:00     35  AgentsVille Comedy Theatre

Date: 2025-06-11
Weather: Sunny (68.0°F)
                           name               start_time  price                    location
0  AgentsVille Museum of Modern Art  2025-06-11 10:00:00+00:00     25  AgentsVille Art District
1        Jazz & Blues Evening       2025-06-11 19:00:00+00:00     45     The Blue Note Lounge

Date: 2025-06-12
Weather: Clear (75.0°F)
                         name               start_time  price              location
0  Shakespeare in the Park     2025-06-12 15:00:00+00:00     30  AgentsVille Park
1  Cooking Masterclass         2025-06-12 10:00:00+00:00     60    Culinary Studio


## And, just for fun!

In [ ]:
# And finally, just for fun, let's narrate the trip.
# No changes needed here.

from project_lib import narrate_my_trip

narrate_my_trip(
    vacation_info=vacation_info,
    itinerary=travel_plan_2,
    client=client,
    model=MODEL,
)


🎙️ Day 1 — June 10, 2025: Your adventure begins in AgentsVille! Start the morning with the AgentsVille Tennis Open at the Sports Club — the perfect warm-up for Yuri. Wind down with a hilarious Comedy Night at Laughs Lounge, guaranteed to bring the laughs!
🎙️ Day 2 — June 11, 2025: A sunny day calls for culture! Explore the AgentsVille Museum of Modern Art — a feast for Hiro's artistic soul. Cap it off with a soulful Jazz & Blues Evening at The Blue Note Lounge.
🎙️ Day 3 — June 12, 2025: Clear skies for a grand finale! Catch Shakespeare in the Park for an unforgettable theatrical experience, then channel your inner chef at the Cooking Masterclass. Perfect endings!


## CONGRATULATIONS! 🎉🥳👏

You have successfully planned a stellar vacation to AgentsVille! Your AI travel agent has demonstrated advanced reasoning techniques, including role-based prompting, chain-of-thought reasoning, ReAct prompting, and feedback loops.

Give yourself a pat on the back for completing this project and completing this course!